In [ ]:
"""
Sliding window attention: 
"""
import numpy as np

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis = -1, keepdims = True))
    return exp_z / np.sum(exp_z, axis = -1, keepdims = True)

def sliding_window_attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray, window_size: int) -> np.ndarray:
    """
    Compute sliding window attention.
    
    Args:
        Q: Query matrix of shape (seq_len, d_k)
        K: Key matrix of shape (seq_len, d_k)
        V: Value matrix of shape (seq_len, d_v)
        window_size: Number of positions to the left and right each query can attend to
    
    Returns:
        Output matrix of shape (seq_len, d_v), rounded to 4 decimal places.
    """
    n, d = Q.shape
    score_matrix = []
    for i, row in enumerate(Q):
        score = row @ K.T / np.sqrt(d)
        left = max(0, i - window_size)
        right = min(i + window_size + 1, n)
        mask = np.zeros(n, dtype = bool)
        mask[left:right] = True
        # score[~mask] = -np.inf
        score = np.where(mask, score, -np.inf)
        score_matrix.append(score)
    score_matrix = np.stack(score_matrix, axis = 0)
    return np.round(softmax(score_matrix) @ V, 4)

Q = np.array([[1.0], [2.0], [3.0]])
K = np.array([[1.0], [2.0], [3.0]])
V = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
result = sliding_window_attention(Q, K, V, window_size=1)
print(result.tolist())

#expected: [[0.2689, 0.7311], [0.8827, 0.9841], [0.9526, 1.0]]


[[0.2689, 0.7311], [0.8827, 0.9841], [0.9526, 1.0]]


In [2]:
"""
Flash Attention v1 - Forward Pass: reducing O(n^2) memory to O(n), 
by computing attention in blocks and using online softmax, 
never materializing the full NxN attention matrix.
"""
import numpy as np

def softmax(z):
	exp_z = np.exp(z - np.max(z, axis = -1, keepdims = True))
	return exp_z / np.sum(exp_z, axis = -1, keepdims = True)

def flash_attention_forward(Q: np.ndarray, K: np.ndarray, V: np.ndarray, 
                           block_size: int = 2) -> np.ndarray:
	"""
	Compute attention output using Flash Attention v1 algorithm.
	
	Args:
		Q: Query matrix (seq_len, d_model)
		K: Key matrix (seq_len, d_model)
		V: Value matrix (seq_len, d_model)
		block_size: Size of blocks for tiled computation
	
	Returns:
		Output matrix (seq_len, d_model)
	"""
	n, d = Q.shape
	post_attn = np.zeros(V.shape)
	# Your code here
	for i in range(0, n, block_size):
		Qi = Q[i:i+block_size, :]
		br = Qi.shape[0]
		O = np.zeros(Qi.shape)
		l = np.zeros((br, 1))
		m = np.full(l.shape, -np.inf)
		for j in range(0, n, block_size):
			Kj = K[j:j+block_size, :]
			Vj = V[j:j+block_size, :]
			score = Qi @ Kj.T / np.sqrt(d)
			m_new = np.maximum(m, np.max(score, axis = -1, keepdims = True))
			factor = np.exp(m - m_new)
			l_new = factor * l + np.sum(np.exp(score - m_new), axis = -1, keepdims = True)
			O = factor * O + np.exp(score - m_new) @ Vj
			m, l = m_new, l_new
		post_attn[i:i+br, :] = O / l
	return post_attn


np.random.seed(42)
Q = np.random.randn(4, 4)
K = np.random.randn(4, 4)
V = np.random.randn(4, 4)
out = flash_attention_forward(Q, K, V, block_size=2)
print(np.round(out[0], 4).tolist())

#expected: [-0.8285, -0.716, -0.4417, 0.6186]


[-0.8285, -0.716, -0.4417, 0.6186]
